# Yousee Piran — Underwater Visibility Scorer

Main pipeline: calibrate → validate → batch score all images.

In [ ]:
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import glob
import csv
import datetime
import re
import os
import sys

sys.path.insert(0, os.path.abspath("."))
from scorer import (
    score_image, reload_calibration, is_scoreable, load_image,
    dark_channel_score, tenengrad_sharpness, blue_dominance,
    uicm_normalized, preprocess_for_metrics, CALIBRATION_PATH,
)

BASE_DATA = os.path.join(os.path.abspath("."), "data")
print("scorer loaded, calibration:", CALIBRATION_PATH)


## Cell 1 — Calibration

Samples 50 images stratified by camera type; writes `calibration.json`.

In [ ]:
# Calibrate: sample 50 images stratified by camera type, write calibration.json.
# Skip this cell if calibration.json already exists and you have not changed the dataset.
import subprocess, sys

result = subprocess.run(
    [sys.executable, os.path.join(os.path.abspath("."), "_calibrate.py")],
    capture_output=True, text=True,
)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print("CALIBRATION FAILED\n", result.stderr[-1000:])
else:
    reload_calibration()
    print("\nCalibration reloaded from:", CALIBRATION_PATH)


## Cell 2 — 15-Image Validation

Compares new scorer against old turbidity baseline (old scores from `plan-scan-fix-validation.md`).

In [ ]:
# 15-image validation — same images used in plan-scan.md Section 6a.
# Shows current NEW scorer output; OLD scores are in plan-scan-fix-validation.md.
FIFTEEN = [
    (1,  "data/train/images/20231016-173101-IPC608_8B64_165.jpg"),
    (2,  "data/train/images/20231002-055401-IPC608_8B64_165.jpg"),
    (3,  "data/train/images/20231018-105601-IPC608_8B64_165.jpg"),
    (4,  "data/train/images/20240401-141501-AIPC608UW_10_167.jpg"),
    (5,  "data/train/images/20230404-120000-C4k0193.jpg"),
    (6,  "data/valid/images/20231202-142401-IPC608_8B64_165.jpg"),
    (7,  "data/valid/images/20231001-102401-IPC608_8B64_165.jpg"),
    (8,  "data/valid/images/20240411-121901-IPC608_8B64_165.jpg"),
    (9,  "data/valid/images/20240505-160901-IPC608_8BC7_166.jpg"),
    (10, "data/valid/images/20230404-160000-C4k0193.jpg"),
    (11, "data/test/images/20240229-132001-IPC608_8B64_165.jpg"),
    (12, "data/test/images/20240306-Video-2-IPC608_8B64_166_frame087.jpg"),
    (13, "data/test/images/20231007-090201-IPC608_8B64_165.jpg"),
    (14, "data/test/images/20240410-120800-Video-AIPC608UW_10_167_frame010.jpg"),
    (15, "data/test/images/20230325-180000-C4k0193.jpg"),
]

OLD_SCORES = {
    1: 54.4, 2: 58.1, 3: 61.1, 4: 56.5, 5: 56.9,
    6: 64.5, 7: 61.9, 8: 63.0, 9: 57.0, 10: 56.6,
    11: 65.2, 12: 62.1, 13: 59.9, 14: 57.1, 15: 54.2,
}

print(f"{'#':>2}  {'File':<55}  {'OLD':>6}  {'NEW':>6}  {'Delta':>7}  Unscoreable?")
print("-" * 100)
clarities = []
for idx, rel_path in FIFTEEN:
    path = os.path.join(os.path.abspath("."), rel_path)
    r = score_image(path)
    old = OLD_SCORES[idx]
    new_c = r["clarity_score"]
    fname = os.path.basename(path)[:54]
    if new_c is None:
        delta_str = "  —"
        new_str = "UNSCOREABLE"
        unscore = r["unscoreable_reason"] or ""
    else:
        delta_str = f"{new_c - old:+7.1f}"
        new_str = f"{new_c:6.1f}"
        unscore = ""
        clarities.append(new_c)
    print(f"{idx:2d}  {fname:<55}  {old:6.1f}  {new_str:>11}  {delta_str}  {unscore}")

if clarities:
    print(f"\nNEW clarity — n={len(clarities)}  range={min(clarities):.1f}–{max(clarities):.1f}"
          f"  spread={max(clarities)-min(clarities):.1f}  std={np.std(clarities):.2f}")


## Cell 3 — 50-Image Full Validation

Stratified sample (seed=137, different from calibration seed=42).

In [ ]:
# 50-image full validation — stratified by camera, seed=137 (differs from calibration seed=42).
CAMERAS = ["IPC608_8B64_165", "IPC608_8BC7_166", "AIPC608UW_10_167", "C4k0193"]

all_images = []
for split in ("train", "valid", "test"):
    all_images += glob.glob(os.path.join(BASE_DATA, split, "images", "*.jpg"))

buckets = {cam: [] for cam in CAMERAS}
other = []
for p in all_images:
    name = os.path.basename(p)
    matched = False
    for cam in CAMERAS:
        if cam in name:
            buckets[cam].append(p)
            matched = True
            break
    if not matched:
        other.append(p)

rng = np.random.default_rng(137)
selected = []
n_per_bucket = 50 // len(CAMERAS)
for cam in CAMERAS:
    pool = sorted(buckets[cam])
    n = min(n_per_bucket, len(pool))
    idx = rng.choice(len(pool), size=n, replace=False)
    selected += [pool[i] for i in sorted(idx)]
remaining = 50 - len(selected)
if remaining > 0 and other:
    pool = sorted(other)
    n = min(remaining, len(pool))
    idx = rng.choice(len(pool), size=n, replace=False)
    selected += [pool[i] for i in sorted(idx)]

results = []
for path in selected:
    r = score_image(path)
    r["path"] = path
    results.append(r)

scored = [r for r in results if r["clarity_score"] is not None]
unscored = [r for r in results if r["clarity_score"] is None]
clarities = [r["clarity_score"] for r in scored]

print(f"n_total={len(results)}  n_scored={len(scored)}  n_unscored={len(unscored)}")
if clarities:
    arr = np.array(clarities)
    print(f"range={arr.min():.1f}–{arr.max():.1f}  spread={arr.max()-arr.min():.1f}"
          f"  std={arr.std():.2f}")
    pcts = np.percentile(arr, [5, 25, 50, 75, 95])
    print(f"p5={pcts[0]:.1f}  p25={pcts[1]:.1f}  median={pcts[2]:.1f}"
          f"  p75={pcts[3]:.1f}  p95={pcts[4]:.1f}")
    print()
    bins = range(0, 101, 10)
    for lo in range(0, 100, 10):
        hi = lo + 10
        count = sum(1 for c in clarities if lo <= c < hi)
        bar = "█" * count
        print(f"  {lo:3d}–{hi:3d} | {bar:<30} {count}")


## Cell 4 — Batch Scorer (Fix 8)

Scores all images; writes `labels/visibility_scores.csv`.

In [ ]:
# Fix 8 — Batch scorer: score all images and write labels/visibility_scores.csv.
# Run once; re-run after re-calibrating.
import csv

LABEL_DIR = os.path.join(os.path.abspath("."), "labels")
os.makedirs(LABEL_DIR, exist_ok=True)
OUT_CSV = os.path.join(LABEL_DIR, "visibility_scores.csv")

all_images = []
for split in ("train", "valid", "test"):
    all_images += sorted(glob.glob(os.path.join(BASE_DATA, split, "images", "*.jpg")))

print(f"Scoring {len(all_images)} images → {OUT_CSV}")

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "image_path", "split", "filename", "timestamp",
        "clarity_score", "turbidity_score", "unscoreable_reason",
        "dcp_raw", "tenengrad_raw", "blue_dom_raw", "uicm_raw",
        "dcp_score", "tenengrad_score", "blue_dom_score", "uicm_score",
        "image_w", "image_h",
    ])
    n_scored = n_unscored = 0
    for path in all_images:
        split = path.split(os.sep)[-3]  # train / valid / test
        fname = os.path.basename(path)
        # Extract timestamp from filename: YYYYMMDD-HHMMSS prefix
        m = re.match(r"(\d{8}-\d{6})", fname)
        ts = m.group(1) if m else ""
        r = score_image(path)
        raw = r.get("metrics_raw", {})
        scr = r.get("metrics_score", {})
        writer.writerow([
            path, split, fname, ts,
            r["clarity_score"], r["turbidity_score"], r["unscoreable_reason"],
            raw.get("dcp"), raw.get("tenengrad"), raw.get("blue_dom"), raw.get("uicm"),
            scr.get("dcp_score"), scr.get("tenengrad_score"),
            scr.get("blue_dom_score"), scr.get("uicm_score"),
            r["image_w"], r["image_h"],
        ])
        if r["clarity_score"] is not None:
            n_scored += 1
        else:
            n_unscored += 1

print(f"Done. scored={n_scored}  unscored={n_unscored}")
print(f"CSV written: {OUT_CSV}")

df = pd.read_csv(OUT_CSV)
scored_df = df[df["clarity_score"].notna()]
if len(scored_df):
    arr = scored_df["clarity_score"].values
    print(f"\nFull dataset clarity — n={len(arr)}  range={arr.min():.1f}–{arr.max():.1f}"
          f"  std={arr.std():.2f}  median={np.median(arr):.1f}")
